---

## ModelCallLimitMiddleware 옵션별 테스트

``ModelCallLimitMiddleware`` 는 에이전트의 **모델(LLM) 호출 횟수**를 추적하고,
한도를 넘으면 실행을 중단합니다.

- 훅: ``before_model`` (한도 확인·조기 종료), ``after_model`` (호출 수 증가)

**참고:** [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

아래 각 섹션은 ``feature/MiddlewareModelCallLimit.py`` 의 ``MiddlewareModelCallLimitAgent`` 로
옵션 하나씩 바꿔 동작을 확인합니다.

**옵션 요약:**

| 옵션 | 기본값 | 역할 |
|:---|:---|:---|
| ``thread_limit`` | ``3`` | **스레드(대화) 전체**에서 허용할 최대 모델 호출 수. ``None`` 이면 제한 없음 |
| ``run_limit`` | ``2`` | **단일 invoke** 안에서 허용할 최대 모델 호출 수. ``None`` 이면 제한 없음 |
| ``exit_behavior`` | ``"end"`` | 한도 초과 시 동작 — ``"end"``: 종료 + 안내 메시지, ``"error"``: 예외 |

**``thread_limit`` vs ``run_limit``**

| 구분 | 범위 | 예시 |
|:---|:---|:---|
| ``run_limit`` | **한 번의** ``invoke()`` | 도구 호출 시 model→tool→model 로 2회 호출 → ``run_limit=1`` 이면 2번째에서 차단 |
| ``thread_limit`` | **같은** ``thread_id`` 의 **여러 번** ``invoke()`` | 1회 invoke 에서 1회 호출 후, 같은 스레드로 다시 invoke → 누적 카운트로 차단 |

``thread_limit`` 테스트에는 ``checkpointer`` + ``thread_id`` 가 필요합니다 (기본 활성화).
한도 초과(``exit_behavior="end"``) 시 ``Model call limits exceeded: ...`` 로 시작하는
``AIMessage`` 가 주입됩니다. ``messages_contain_limit_exceeded()`` 로 확인합니다.

In [1]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

from feature.MiddlewareModelCallLimit import (
    MiddlewareModelCallLimitAgent,
    ModelCallLimitExceededError,
    messages_contain_limit_exceeded,
)

### 1. 기본 설정 — 한도 내 정상 응답

기본 ``thread_limit=3`` / ``run_limit=2`` / ``exit_behavior="end"``.
단순 질문 1회는 모델 호출 1번이므로 **정상 응답**이어야 합니다.

In [2]:
agent_default = MiddlewareModelCallLimitAgent()

result = agent_default.invoke(
    inputs={"messages": [HumanMessage(content="Hello!")]},
    config=RunnableConfig(configurable={"thread_id": "mcl-default"}),
)

assert result is not None
assert not messages_contain_limit_exceeded(result["messages"]), "한도 내 — 차단 메시지 없어야 함"
print("✓ 기본 설정 — 정상 응답:", result["messages"][-1].content[:120])


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Hello! How can I assist you today?

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1
✓ 기본 설정 — 정상 응답: Hello! How can I assist you today?


### 2. ``run_limit=1`` — 단일 invoke 내 2번째 모델 호출 차단

도구 호출 흐름은 보통 **model → tool → model** (2회)입니다.
``run_limit=1`` 이면 2번째 ``before_model`` 에서 한도 초과 → ``exit_behavior="end"`` 로 종료됩니다.

In [3]:
agent_run_limit = MiddlewareModelCallLimitAgent(
    thread_limit=None,
    run_limit=1,
    exit_behavior="end",
)

result = agent_run_limit.invoke(
    inputs={"messages": [HumanMessage(content="What's the weather in Seoul?")]},
    config=RunnableConfig(configurable={"thread_id": "mcl-run-limit"}),
)

assert messages_contain_limit_exceeded(result["messages"]), "run_limit=1 — 한도 초과 AIMessage 기대"
limit_msg = next(
    m.content for m in result["messages"] if messages_contain_limit_exceeded([m])
)
print("✓ run_limit=1 — 차단 메시지:", limit_msg)


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_yHfHQxuSIOZTbikI1176FatF)
 Call ID: call_yHfHQxuSIOZTbikI1176FatF
  Args:
    city: Seoul

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
jump_to:
end
================================== Ai Message ==================================

Model call limits exceeded: run limit (1/1)
✓ run_limit=1 — 차단 메시지: Model call limits exceeded:

### 3. ``thread_limit=1`` — 같은 스레드 2번째 invoke 차단

``thread_model_call_count`` 는 **스레드 전체**에 누적됩니다.
1회 invoke 후 같은 ``thread_id`` 로 다시 호출하면, 2번째 invoke 의 ``before_model`` 에서 차단됩니다.

In [4]:
agent_thread_limit = MiddlewareModelCallLimitAgent(
    thread_limit=1,
    run_limit=None,
    exit_behavior="end",
)

cfg = RunnableConfig(configurable={"thread_id": "mcl-thread-limit"})

result1 = agent_thread_limit.invoke(
    inputs={"messages": [HumanMessage(content="Hi there!")]},
    config=cfg,
)
assert not messages_contain_limit_exceeded(result1["messages"]), "1회 invoke — 정상"

result2 = agent_thread_limit.invoke(
    inputs={"messages": [HumanMessage(content="One more question.")]},
    config=cfg,
)
assert messages_contain_limit_exceeded(result2["messages"]), "2회 invoke — thread_limit=1 차단"
print("✓ thread_limit=1 — 2번째 invoke 차단 확인")


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Hello! How can I assist you today?

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1

🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
jump_to:
end
================================== Ai Message ==================================

Model call limits exceeded: thread limit (1/1)
✓ thread_limit=1 — 2번째 invoke 차단 확인


### 4. ``exit_behavior="end"`` — 조기 종료 + 안내 메시지

한도 초과 시 예외 없이 **에이전트 실행을 끝내고**, 아래 형식의 ``AIMessage`` 를 주입합니다.

```
Model call limits exceeded: run limit (1/1)
```

In [5]:
agent_end = MiddlewareModelCallLimitAgent(
    thread_limit=None,
    run_limit=1,
    exit_behavior="end",
)

result = agent_end.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Tokyo?")]},
    config=RunnableConfig(configurable={"thread_id": "mcl-exit-end"}),
)

limit_msg = next(
    m.content for m in result["messages"] if messages_contain_limit_exceeded([m])
)
assert "run limit" in limit_msg
print("✓ exit_behavior=end —", limit_msg)


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Sm22quVRFxkhEwaJOYziKd8l)
 Call ID: call_Sm22quVRFxkhEwaJOYziKd8l
  Args:
    city: Tokyo

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Tokyo!

🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
jump_to:
end
================================== Ai Message ==================================

Model call limits exceeded: run limit (1/1)
✓ exit_behavior=end — Model call limits exceeded: r

### 5. ``exit_behavior="error"`` — ``ModelCallLimitExceededError`` 예외

한도 초과 시 ``ModelCallLimitExceededError`` 가 발생합니다.
예외 객체에 ``thread_count`` / ``run_count`` / ``thread_limit`` / ``run_limit`` 속성이 있습니다.

In [6]:
agent_error = MiddlewareModelCallLimitAgent(
    thread_limit=None,
    run_limit=1,
    exit_behavior="error",
)

try:
    agent_error.invoke(
        inputs={"messages": [HumanMessage(content="Weather in Paris?")]},
        config=RunnableConfig(configurable={"thread_id": "mcl-exit-error"}),
    )
    raise AssertionError("run_limit=1 초과 시 ModelCallLimitExceededError 기대")
except ModelCallLimitExceededError as e:
    print(f"✓ exit_behavior=error — {type(e).__name__}: {e}")
    print(f"  run_count={e.run_count}, run_limit={e.run_limit}")


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_5Es62nLc1I3s8mVS83baPvIV)
 Call ID: call_5Es62nLc1I3s8mVS83baPvIV
  Args:
    city: Paris

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Paris!
✓ exit_behavior=error — ModelCallLimitExceededError: Model call limits exceeded: run limit (1/1)
  run_count=1, run_limit=1


### 6. ``thread_limit`` + ``run_limit`` 동시 설정 (기본값)

원본 예제와 동일하게 ``thread_limit=3`` / ``run_limit=2``.
짧은 대화 1회는 두 한도 모두 여유가 있어 정상 동작합니다.

In [7]:
agent_both = MiddlewareModelCallLimitAgent(
    thread_limit=3,
    run_limit=2,
    exit_behavior="end",
)

result = agent_both.invoke(
    inputs={"messages": [HumanMessage(content="What's the weather in Seoul?")]},
    config=RunnableConfig(configurable={"thread_id": "mcl-both-limits"}),
)

if messages_contain_limit_exceeded(result["messages"]):
    print("✓ thread=3/run=2 — run_limit(2)에 걸려 차단 (도구 호출 2회 이상)")
else:
    print("✓ thread=3/run=2 — run_limit 내 정상 완료:", result["messages"][-1].content[:120])


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_ZSHNWDcM4CmLQr3TLWvKM5YI)
 Call ID: call_ZSHNWDcM4CmLQr3TLWvKM5YI
  Args:
    city: Seoul

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!

🔄 Node: 

### 7. ``thread_limit`` 만 설정 — ``run_limit=None``

단일 invoke 내 호출 수는 제한하지 않고, **스레드 누적**만 제한합니다.

In [8]:
agent_thread_only = MiddlewareModelCallLimitAgent(
    thread_limit=2,
    run_limit=None,
    exit_behavior="end",
)

cfg = RunnableConfig(configurable={"thread_id": "mcl-thread-only"})

for i in range(2):
    r = agent_thread_only.invoke(
        inputs={"messages": [HumanMessage(content=f"Message {i+1}")]},
        config=cfg,
    )
    blocked = messages_contain_limit_exceeded(r["messages"])
    print(f"  invoke {i+1}: {'차단' if blocked else '정상'}")

result3 = agent_thread_only.invoke(
    inputs={"messages": [HumanMessage(content="Message 3")]},
    config=cfg,
)
assert messages_contain_limit_exceeded(result3["messages"]), "thread_limit=2 — 3번째 invoke 차단"
print("✓ thread_limit=2, run_limit=None — 3번째 invoke 차단 확인")


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

It looks like you've entered "Message 1." How can I assist you with that? If you have a specific request or question, please let me know!

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1
  invoke 1: 정상

🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

You've entered "Message 2." If you have a specific question or if there's something you'd like to discuss or inquire about, please provide more details!

🔄 Node: ModelCallLimitMiddleware.after_mode

### 8. ``run_limit`` 만 설정 — ``thread_limit=None``

스레드 누적은 제한하지 않고, **한 번의 invoke** 안에서만 제한합니다.
같은 스레드로 여러 번 invoke 해도 invoke 마다 ``run_model_call_count`` 는 0부터 다시 셉니다.

In [9]:
agent_run_only = MiddlewareModelCallLimitAgent(
    thread_limit=None,
    run_limit=1,
    exit_behavior="end",
)

cfg = RunnableConfig(configurable={"thread_id": "mcl-run-only"})

for i in range(2):
    r = agent_run_only.invoke(
        inputs={"messages": [HumanMessage(content="Say hi briefly.")]},
        config=cfg,
    )
    blocked = messages_contain_limit_exceeded(r["messages"])
    print(f"  invoke {i+1} (단순 질문): {'차단' if blocked else '정상'}")

print("✓ run_limit=1, thread_limit=None — invoke마다 run 카운트 리셋, 단순 질문은 매번 정상")


🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Hello!

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
1
run_model_call_count:
1
  invoke 1 (단순 질문): 정상

🔄 Node: ModelCallLimitMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Hi!

🔄 Node: ModelCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
thread_model_call_count:
2
run_model_call_count:
1
  invoke 2 (단순 질문): 정상
✓ run_limit=1, thread_limit=None — invoke마다 run 카운트 리셋, 단순 질문은 매번 정상


### 9. (검증) 잘못된 설정 — ``ValueError``

``thread_limit`` 와 ``run_limit`` 를 **둘 다** ``None`` 으로 두면
미들웨어 생성 시 ``ValueError`` 가 납니다.

In [10]:
from feature.MiddlewareModelCallLimit import make_model_call_limit_middleware

try:
    make_model_call_limit_middleware(thread_limit=None, run_limit=None)
    raise AssertionError("둘 다 None이면 ValueError 기대")
except ValueError as e:
    print("✓ 잘못된 설정 — ValueError:", e)

✓ 잘못된 설정 — ValueError: At least one limit must be specified (thread_limit or run_limit)
